# IA GENERATIVA PARA LA GESTIÓN INDUSTRIAL - CERTAMEN 1
## Grupo 8
### Members: Renato Fernández, Vicente Vargas y Matías González

## 1. Base

Comenzaremos creando la estructura principal de nuestra aplicación utilizando Streamlit. Incorporaremos un espacio para subir fotografías de residuos y una sección con información sobre el modelo seleccionado. Luego, conectaremos esta interfaz con YOLO para realizar las clasificaciones y agregar recomendaciones mediante inteligencia artificial.

In [3]:
%%writefile app.py
# Importar las librerías
import streamlit as st
from PIL import Image

# Configurar la aplicación
st.set_page_config(page_title="Clasificación de residuos",page_icon="♻️",layout="wide")

st.title("Clasificación de residuos")
st.write("Sube una fotografía para identificar el material y recibir recomendaciones de reciclaje.")

# Crear las pestañas
tab1,tab2=st.tabs(["Clasificar residuo","Sobre el modelo"])

with tab1:
    # Permitir cargar una fotografía
    archivo=st.file_uploader("Selecciona una imagen",type=["jpg","jpeg","png"])

    if archivo is not None:
        imagen=Image.open(archivo).convert("RGB")
        st.image(imagen,caption="Fotografía cargada",width=400)

Overwriting app.py


## 2. Clasificación de residuos con YOLO

Incorporaremos el modelo YOLO que entrenamos anteriormente para identificar el material de las fotografías cargadas por el usuario. La aplicación utilizará los pesos guardados del modelo para realizar las predicciones y mostrar tanto la categoría identificada como su nivel de confianza.

In [5]:
%%writefile -a app.py
# Importar YOLO
from ultralytics import YOLO

# Cargar el modelo entrenado
@st.cache_resource
def cargar_modelo():
    return YOLO("modelos/yolo_residuos.pt")

with tab1:
    if archivo is not None:
        # Clasificar la imagen
        modelo=cargar_modelo()
        resultado=modelo(imagen,verbose=False)[0]

        # Obtener la clase y su confianza
        clase=resultado.names[resultado.probs.top1]
        confianza=resultado.probs.top1conf.item()

        # Mostrar los resultados
        st.subheader("Resultado de la clasificación")
        col1,col2=st.columns(2)

        with col1:
            st.metric("Material identificado",clase)

        with col2:
            st.metric("Confianza",f"{confianza:.1%}")

Appending to app.py


## 3. Probabilidades de clasificación

Además de mostrar la categoría identificada y su confianza, incorporaremos un gráfico con las probabilidades de los cinco materiales. Esto permitirá observar las alternativas consideradas por el modelo y reconocer cuándo existen posibles confusiones entre categorías.

In [7]:
%%writefile -a app.py
import pandas as pd

with tab1:
    if archivo is not None:
        # Obtener las probabilidades de las cinco clases
        probabilidades=resultado.probs.data.cpu().numpy()

        # Organizar los resultados en una tabla
        tabla=pd.DataFrame({
            "Material":[resultado.names[i] for i in range(len(probabilidades))],
            "Probabilidad (%)":(probabilidades*100).round(1)
        })

        # Mostrar las probabilidades
        st.subheader("Probabilidades por material")
        st.bar_chart(tabla.set_index("Material"))
        st.dataframe(tabla,hide_index=True,use_container_width=True)

Appending to app.py


## 4. Advertencias de clasificación

Incorporaremos una advertencia cuando la confianza del modelo sea inferior al 70%, para informar al usuario que la clasificación podría ser incorrecta. Este porcentaje se utilizará como un criterio práctico de referencia y no garantiza que las predicciones superiores al 70% sean correctas.

In [9]:

%%writefile -a app.py

with tab1:
    if archivo is not None:
        # Advertir cuando la confianza sea baja
        if confianza < 0.70:
            st.warning("La confianza de esta predicción es baja. Se recomienda verificar el material antes de separarlo para reciclar.")

        # Recordar las limitaciones de la clasificación
        st.caption("Esta herramienta es de apoyo educativo. Sus resultados pueden contener errores y no reemplazan la revisión humana.")


Appending to app.py


## 5. Integración con IA generativa

Incorporaremos Gemini para generar recomendaciones sobre cómo separar y reciclar los residuos identificados por YOLO. La aplicación enviará al modelo de lenguaje la categoría detectada y su nivel de confianza, con el objetivo de entregar recomendaciones útiles para el usuario. Para conectarnos de forma segura, utilizaremos una API Key almacenada fuera del código principal.

In [11]:
%%writefile .gitignore
# Archivos privados
.streamlit/secrets.toml
.env

# Archivos que no necesitamos publicar
datos/
runs/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [12]:
# Comprobar que la API Key está guardada
import tomllib
with open(".streamlit/secrets.toml","rb") as f:
    secretos=tomllib.load(f)
print("API Key configurada:",bool(secretos.get("GEMINI_API_KEY")))

API Key configurada: True


### 5.4. Generación de recomendaciones

Conectaremos nuestra aplicación con Gemini para entregar recomendaciones de reciclaje a partir del material identificado por YOLO y su nivel de confianza. El usuario podrá solicitar estas recomendaciones mediante un botón. Además, indicaremos las limitaciones de la clasificación para evitar entregar instrucciones demasiado categóricas cuando el modelo tenga dudas.

In [14]:
%%writefile -a app.py
# Importar Gemini
from google import genai

with tab1:
    if archivo is not None:
        # Generar una recomendación al presionar el botón
        if st.button("Obtener recomendación de reciclaje"):
            prompt=f"""
            Eres un asistente especializado en reciclaje de residuos.

            YOLO identificó el material: {clase}.
            Confianza de la clasificación: {confianza:.1%}.

            Entrega recomendaciones breves y prácticas en español sobre
            cómo separar y preparar este material para reciclarlo.
            Si la confianza es inferior al 70%, advierte que es
            importante verificar el material.
            No asegures que el residuo es reciclable sin conocer
            su composición ni las condiciones locales de reciclaje.
            """

            # Consultar la API de Gemini
            try:
                with st.spinner("Generando recomendación..."):
                    cliente=genai.Client(api_key=st.secrets["GEMINI_API_KEY"])
                    respuesta=cliente.models.generate_content(
                        model="gemini-3.5-flash-lite",
                        contents=prompt
                    )

                st.subheader("Recomendación de reciclaje")
                if respuesta.text:
                    st.write(respuesta.text)
                else:
                    st.warning("No se obtuvo una recomendación.")

            except Exception:
                st.error("No fue posible consultar Gemini. Revisa la conexión y la API Key.")

Appending to app.py


## 6. Información y desempeño del modelo

Incorporaremos información sobre el dataset utilizado, los modelos entrenados y sus resultados de validación. Esto permitirá que los usuarios conozcan el rendimiento de nuestro clasificador y comprendan algunas de sus limitaciones.

In [16]:
%%writefile -a app.py
with tab2:
    # Información del dataset
    st.subheader("Datos utilizados")
    st.write("Utilizamos el dataset RealWaste, seleccionando 3.092 imágenes de cinco materiales.")
    st.write("El 80% se utilizó para entrenamiento y el 20% para validación.")

    # Mostrar el rendimiento de los modelos
    st.subheader("Comparación de modelos")
    col1,col2=st.columns(2)

    with col1:
        st.metric("EfficientNet-B0", "87,3%")

    with col2:
        st.metric("YOLO26n-cls", "91,5%")
    st.caption("Accuracy balanceada obtenida durante la validación.")

    # Resultados del modelo seleccionado
    st.subheader("Rendimiento de YOLO")
    st.write("Accuracy: 91,0%")
    st.write("Accuracy balanceada: 91,5%")
    st.write("Imágenes clasificadas correctamente: 564 de 620.")
    st.info("Estos resultados corresponden a validación. El rendimiento puede variar al utilizar fotografías nuevas.")

Appending to app.py
